# 04 最终九类因子回测

回测函数集中在 `src/treasury_futures/factors/cicc_close_session_reverse/backtest.py`；本 Notebook 只保留筛选配置、运行和结果展示。


In [1]:
from pathlib import Path
import sys

_PROJECT_CANDIDATES = (Path.cwd(), *Path.cwd().parents)
PROJECT_ROOT = next(
    (candidate for candidate in _PROJECT_CANDIDATES if (candidate / "pyproject.toml").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("找不到项目根目录 pyproject.toml；请从本项目内启动 Jupyter。")
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from IPython.display import display
from treasury_futures.factors.cicc_close_session_reverse.backtest import *  # noqa: F403


In [2]:

# 回测信号筛选配置：None 表示该维度不过滤。
# factor_ids 示例：frozenset({"closing_capital_oi_accel_h3"})
# categories 示例：frozenset({"2. 尾盘资金进场"})
# roll_statuses 示例：frozenset({"CROSSOVER","ACTIVE", "WATCH","NORMAL"})
signal_filter = SignalFilterConfig(
    exclude_blocked=True,       # 默认排除 block_signal=True 的移仓/数据异常信号
    factor_ids=None,            # 只回测指定 factor_id；None=全部
    categories=None,            # 只回测指定类别；None=全部
    roll_statuses={"ACTIVE","WATCH","NORMAL"},         # 只回测指定移仓状态；None=全部未被 block 的状态
    signal_start_date=None,     # 例如 "2022-01-01"
    signal_end_date=None,       # 例如 "2025-12-31"
)

result = run_backtest(signal_filter=signal_filter)
print(f'backtest workbook: {FACTOR_OUTPUT_DIR / BACKTEST_WORKBOOK}')


信号筛选：输入=126，保留=94，筛除=32
VWAP amount unit detected: multiplier=10,000, median_ratio=10,000.000
backtest workbook: E:\CodeBase\TaiKangAM\TreasuryFutures\Factor\notebooks\CICC_CloseSession_Reverse\CTreasuryFutures\factors\cicc_close_session_reverse\output\final_9_category_backtest.xlsx


## 完整逐笔交易
逐笔展示每个语义因子的信号日、入场日、出场日、价格与扣费后 PnL。

In [3]:
display(result.trades)

,factor_id,signal_date,trade_date,roll_status,block_signal,entry_date,exit_date,hold_days,direction,entry_price,exit_price,pnl_price,pnl_bp,pnl_bp_net,display_name,category
0,trend_contrarian_short_cover_h3,2019-03-15,2019-03-15,NORMAL,False,2019-03-18,2019-03-20,3,-1,89.759715,89.624377,0.135337,15.077737,14.077737,趋势逆势博弈：空平主导,3. 趋势逆势博弈
1,trend_contrarian_short_cover_h3,2019-05-28,2019-05-28,NORMAL,False,2019-05-29,2019-05-31,3,-1,89.497870,89.684572,-0.186702,-20.861035,-21.861035,趋势逆势博弈：空平主导,3. 趋势逆势博弈
2,trend_contrarian_drop_atr_h3,2019-06-10,2019-06-10,NORMAL,False,2019-06-11,2019-06-13,3,-1,90.022839,90.063925,-0.041087,-4.564016,-5.564016,趋势逆势博弈：强跌ATR中档,3. 趋势逆势博弈
3,trend_contrarian_short_cover_h3,2019-06-21,2019-06-21,NORMAL,False,2019-06-24,2019-06-26,3,-1,90.159867,89.994531,0.165336,18.338102,17.338102,趋势逆势博弈：空平主导,3. 趋势逆势博弈
4,closing_capital_oi_accel_h3,2019-11-01,2019-11-01,NORMAL,False,2019-11-04,2019-11-06,3,-1,90.411625,90.851392,-0.439767,-48.640550,-49.640550,尾盘资金进场：持仓加速三日,2. 尾盘资金进场
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
89,trend_contrarian_drop_atr_h3,2025-12-05,2025-12-05,NORMAL,False,2025-12-08,2025-12-10,3,-1,107.824395,108.005541,-0.181146,-16.800071,-17.800071,趋势逆势博弈：强跌ATR中档,3. 趋势逆势博弈
90,trend_contrarian_drop_atr_h3,2026-04-14,2026-04-14,NORMAL,False,2026-04-15,2026-04-17,3,-1,108.344943,108.610298,-0.265356,-24.491734,-25.491734,趋势逆势博弈：强跌ATR中档,3. 趋势逆势博弈
91,closing_capital_oi_accel_h3,2026-06-05,2026-06-05,NORMAL,False,2026-06-08,2026-06-10,3,-1,109.152143,108.835000,0.317143,29.055174,28.055174,尾盘资金进场：持仓加速三日,2. 尾盘资金进场
92,closing_capital_oi_accel_h5,2026-06-05,2026-06-05,NORMAL,False,2026-06-08,2026-06-12,5,-1,109.152143,108.820000,0.332143,30.429403,29.429403,尾盘资金进场：持仓加速五日,2. 尾盘资金进场


## 终端日历不足信号审计
只允许样本末端无法取得 T+持有期出场日的信号进入本表；其他日历或分钟窗口问题仍直接报错。

In [4]:
display(result.dropped_signals)

,signal_date,trade_date,roll_status,block_signal,factor_id,display_name,category,hold_days,direction,signal,reason,signal_position,required_entry_position,required_exit_position,available_last_position,available_last_date
0,2019-02-19,2019-02-19,CROSSOVER,True,range_reversal_tval_hedge_h7,震荡市反转：成交额区间对冲,1. 震荡市反转,7,short,-1,blocked_by_roll_monitor;roll_status_not_selected,NaN,NaN,NaN,NaN,NaN
1,2019-08-08,2019-08-08,ACTIVE,True,closing_capital_oi_accel_h3,尾盘资金进场：持仓加速三日,2. 尾盘资金进场,3,short,-1,blocked_by_roll_monitor,NaN,NaN,NaN,NaN,NaN
2,2019-08-08,2019-08-08,ACTIVE,True,closing_capital_oi_accel_h5,尾盘资金进场：持仓加速五日,2. 尾盘资金进场,5,short,-1,blocked_by_roll_monitor,NaN,NaN,NaN,NaN,NaN
3,2019-08-08,2019-08-08,ACTIVE,True,closing_capital_oi_accel_h7,尾盘资金进场：持仓加速七日,2. 尾盘资金进场,7,short,-1,blocked_by_roll_monitor,NaN,NaN,NaN,NaN,NaN
4,2020-02-12,2020-02-12,ACTIVE,True,closing_capital_oi_accel_h3,尾盘资金进场：持仓加速三日,2. 尾盘资金进场,3,short,-1,blocked_by_roll_monitor,NaN,NaN,NaN,NaN,NaN
5,2020-02-12,2020-02-12,ACTIVE,True,closing_capital_oi_accel_h5,尾盘资金进场：持仓加速五日,2. 尾盘资金进场,5,short,-1,blocked_by_roll_monitor,NaN,NaN,NaN,NaN,NaN
6,2020-02-12,2020-02-12,ACTIVE,True,closing_capital_oi_accel_h7,尾盘资金进场：持仓加速七日,2. 尾盘资金进场,7,short,-1,blocked_by_roll_monitor,NaN,NaN,NaN,NaN,NaN
7,2021-05-14,2021-05-14,ACTIVE,True,trend_contrarian_drop_atr_h3,趋势逆势博弈：强跌ATR中档,3. 趋势逆势博弈,3,short,-1,blocked_by_roll_monitor,NaN,NaN,NaN,NaN,NaN
8,2021-08-12,2021-08-12,ACTIVE,True,trend_contrarian_drop_rv_event_h3,趋势逆势博弈：强跌高波动近事件,3. 趋势逆势博弈,3,short,-1,blocked_by_roll_monitor,NaN,NaN,NaN,NaN,NaN
9,2023-02-14,2023-02-14,ACTIVE,True,closing_capital_oi_accel_h3,尾盘资金进场：持仓加速三日,2. 尾盘资金进场,3,short,-1,blocked_by_roll_monitor,NaN,NaN,NaN,NaN,NaN


## 因子统计
按中文因子名称汇总胜率、均值与累计收益。

In [5]:
display(result.factor_stats)

,factor_id,display_name,trade_count,win_rate,avg_pnl_bp,total_pnl_bp
0,closing_capital_oi_accel_h3,尾盘资金进场：持仓加速三日,11,0.727273,8.923346,98.156808
1,closing_capital_oi_accel_h5,尾盘资金进场：持仓加速五日,11,0.636364,7.911800,87.029795
2,closing_capital_oi_accel_h7,尾盘资金进场：持仓加速七日,11,0.545455,6.753873,74.292601
3,d_segment_consistency_h5,D段一致性：尾盘方向确认,7,0.428571,-9.414529,-65.901706
4,range_reversal_donch_hedge_event_h7,震荡市反转：唐奇安区间远事件,5,0.800000,31.412896,157.064481
5,range_reversal_tval_hedge_h7,震荡市反转：成交额区间对冲,8,0.750000,23.870383,190.963067
6,trend_contrarian_drop_atr_h3,趋势逆势博弈：强跌ATR中档,11,0.545455,2.164536,23.809901
7,trend_contrarian_drop_rv_event_h3,趋势逆势博弈：强跌高波动近事件,8,0.750000,11.289167,90.313336
8,trend_contrarian_short_cover_h3,趋势逆势博弈：空平主导,22,0.727273,5.485798,120.687547


## 类别统计、总体统计、年度统计
总体统计固定展示输入信号数、可执行交易数、终端丢弃数；类别与年度口径用于导师快速复核组合贡献。

In [6]:
display(result.category_stats)
display(result.overall)
display(result.yearly)

,category,trade_count,win_rate,avg_pnl_bp,total_pnl_bp
0,1. 震荡市反转,13,0.769231,26.771350,348.027548
1,2. 尾盘资金进场,33,0.636364,7.863006,259.479204
2,3. 趋势逆势博弈,41,0.682927,5.727092,234.810785
3,4. D段一致性,7,0.428571,-9.414529,-65.901706


,metric,value
0,input_signal_count,126.000000
1,selected_signal_count,94.000000
2,filter_dropped_count,32.000000
3,executable_trade_count,94.000000
4,dropped_signal_count,32.000000
5,win_rate,0.659574
6,avg_pnl_bp,8.259743
7,total_pnl_bp,776.415830
8,max_drawdown_bp,-346.651344


,year,trade_count,pnl_bp_net
0,2019,8,-195.527584
1,2020,15,271.504799
2,2021,12,178.556646
3,2022,8,112.625289
4,2023,5,110.549164
5,2024,18,-75.166904
6,2025,24,326.652536
7,2026,4,47.221884


## 综合回测图
价格和九个因子信号、累积净 PnL、回撤使用同一时间轴联动展示。颜色表示四大类别，形状区分信号组。

第二张图使用单份仓口径：同一 `signal_date` 无论出现几个信号都只开一份空仓，持有期取同日可执行信号中的最大 `hold_days`，往返成本只扣一次；不同信号日产生的仓位仍允许重叠。图中同时保留原口径曲线作为对照；后面的“持仓时间与策略年化绩效”表仍沿用原逐信号口径。

In [7]:
result.combined_figure.show(config={'responsive': True, 'displaylogo': False})
result.same_day_single_position_figure.show(config={'responsive': True, 'displaylogo': False})

## 持仓时间与策略年化绩效

持仓比例使用“至少有一笔仓位存续的交易日数 / 首次入场至最后退出之间的总交易日数”，重叠交易只计一次；同时报告累计单笔持仓天数，以观察重叠仓位。

默认只统计 `POST_OPEN_SWITCH = 2020-07-20` 之后的信号，与综合回测图的展示区间一致。收益口径与本 Notebook 现有组合曲线保持一致：每笔交易的 `pnl_bp_net` 在退出日确认，同日多笔交易直接相加，非退出日收益记为 0；按 252 个交易日年化，无风险利率取 0。该口径不是逐日盯市，也没有对重叠仓位进行资金归一化。

In [8]:
TRADING_DAYS_PER_YEAR = 252
PERFORMANCE_START_DATE = POST_OPEN_SWITCH  # 与综合回测图一致；设为 None 可统计全部历史

trades_for_metrics = result.trades.copy()
trades_for_metrics['signal_date'] = pd.to_datetime(trades_for_metrics['signal_date']).dt.normalize()
if PERFORMANCE_START_DATE is not None:
    performance_start_date = pd.Timestamp(PERFORMANCE_START_DATE).normalize()
    trades_for_metrics = trades_for_metrics[trades_for_metrics['signal_date'] >= performance_start_date].copy()
if trades_for_metrics.empty:
    raise BacktestInputError('没有可执行交易，无法计算持仓比例和年化绩效。')

# 统一日期格式，并以第一笔入场到最后一笔退出作为策略实际回测区间。
trades_for_metrics['entry_date'] = pd.to_datetime(trades_for_metrics['entry_date']).dt.normalize()
trades_for_metrics['exit_date'] = pd.to_datetime(trades_for_metrics['exit_date']).dt.normalize()
first_entry_date = trades_for_metrics['entry_date'].min()
last_exit_date = trades_for_metrics['exit_date'].max()

# 使用本次回测撮合时已经生成的真实交易日历，不重复读取 Parquet。
daily_calendar = pd.Series(pd.to_datetime(result.trading_calendar)).dt.normalize()
backtest_calendar = pd.DatetimeIndex(
    daily_calendar[(daily_calendar >= first_entry_date) & (daily_calendar <= last_exit_date)]
    .drop_duplicates()
    .sort_values()
)
if backtest_calendar.empty:
    raise BacktestInputError('首次入场和最后退出之间没有可用交易日历。')

# 去重后的有仓交易日：任意一笔交易满足 entry_date <= 当日 <= exit_date 即认为当日有仓。
held_trading_mask = pd.Series(False, index=backtest_calendar)
for trade in trades_for_metrics.itertuples(index=False):
    held_trading_mask |= (backtest_calendar >= trade.entry_date) & (backtest_calendar <= trade.exit_date)

total_trading_days = int(len(backtest_calendar))
held_trading_days = int(held_trading_mask.sum())
trading_day_exposure = held_trading_days / total_trading_days

# 累计单笔持仓天数会重复计算重叠仓位，因此可能高于总交易日数。
summed_position_days = int(trades_for_metrics['hold_days'].sum())
gross_position_day_ratio = summed_position_days / total_trading_days

# 自然日口径包含隔夜、周末和节假日，同样对重叠仓位去重。
calendar_dates = pd.date_range(first_entry_date, last_exit_date, freq='D')
held_calendar_mask = pd.Series(False, index=calendar_dates)
for trade in trades_for_metrics.itertuples(index=False):
    held_calendar_mask |= (calendar_dates >= trade.entry_date) & (calendar_dates <= trade.exit_date)
total_calendar_days = int(len(calendar_dates))
held_calendar_days = int(held_calendar_mask.sum())
calendar_day_exposure = held_calendar_days / total_calendar_days

# 沿用当前回测口径：在 exit_date 汇总已经扣除交易成本的净 PnL，并补齐无退出交易的 0 收益日。
exit_day_pnl_bp = trades_for_metrics.groupby('exit_date')['pnl_bp_net'].sum()
daily_strategy_returns = pd.DataFrame({'trade_date': backtest_calendar})
daily_strategy_returns['pnl_bp_net'] = exit_day_pnl_bp.reindex(backtest_calendar, fill_value=0.0).to_numpy()
daily_strategy_returns['daily_return'] = daily_strategy_returns['pnl_bp_net'] / 10_000.0
if (daily_strategy_returns['daily_return'] <= -1).any():
    raise BacktestInputError('存在单日收益率 <= -100%，无法计算复合收益。')

daily_strategy_returns['equity_curve'] = (1.0 + daily_strategy_returns['daily_return']).cumprod()
daily_strategy_returns['running_peak'] = daily_strategy_returns['equity_curve'].cummax().clip(lower=1.0)
daily_strategy_returns['drawdown'] = daily_strategy_returns['equity_curve'] / daily_strategy_returns['running_peak'] - 1.0

daily_return = daily_strategy_returns['daily_return']
total_return = float(daily_strategy_returns['equity_curve'].iloc[-1] - 1.0)
annual_return = float((1.0 + total_return) ** (TRADING_DAYS_PER_YEAR / total_trading_days) - 1.0)
daily_volatility = float(daily_return.std(ddof=1)) if len(daily_return) > 1 else np.nan
annual_volatility = daily_volatility * np.sqrt(TRADING_DAYS_PER_YEAR)
sharpe = (float(daily_return.mean()) / daily_volatility * np.sqrt(TRADING_DAYS_PER_YEAR)) if daily_volatility > 0 else np.nan
max_drawdown = float(daily_strategy_returns['drawdown'].min())
calmar = annual_return / abs(max_drawdown) if max_drawdown < 0 else np.nan

# 保留原始数值，方便后续程序化使用。
strategy_metrics_raw = {
    'first_entry_date': first_entry_date,
    'last_exit_date': last_exit_date,
    'total_calendar_days': total_calendar_days,
    'total_trading_days': total_trading_days,
    'signal_day_count': int(trades_for_metrics['signal_date'].nunique()),
    'trade_count': len(trades_for_metrics),
    'summed_position_days': summed_position_days,
    'held_trading_days': held_trading_days,
    'trading_day_exposure': trading_day_exposure,
    'gross_position_day_ratio': gross_position_day_ratio,
    'held_calendar_days': held_calendar_days,
    'calendar_day_exposure': calendar_day_exposure,
    'total_net_pnl_bp': float(daily_strategy_returns['pnl_bp_net'].sum()),
    'total_return': total_return,
    'annual_return': annual_return,
    'annual_volatility': annual_volatility,
    'sharpe': sharpe,
    'max_drawdown': max_drawdown,
    'calmar': calmar,
}

strategy_metrics = pd.DataFrame([
    {'指标': '首次入场日期', '数值': first_entry_date.strftime('%Y-%m-%d'), '说明': '策略实际开始持仓'},
    {'指标': '最后退出日期', '数值': last_exit_date.strftime('%Y-%m-%d'), '说明': '策略最后一笔交易退出'},
    {'指标': '总自然日时长', '数值': f'{total_calendar_days:,} 天', '说明': '首次入场至最后退出，首尾均包含'},
    {'指标': '总交易日时长', '数值': f'{total_trading_days:,} 天', '说明': '同一区间内的真实交易日数量'},
    {'指标': '纳入信号日数', '数值': f"{trades_for_metrics['signal_date'].nunique():,} 天", '说明': '去重后的 signal_date 数量'},
    {'指标': '完成交易数', '数值': f'{len(trades_for_metrics):,} 笔', '说明': '一个信号日可由多个因子产生多笔交易'},
    {'指标': '累计单笔持仓天数', '数值': f'{summed_position_days:,} 仓位日', '说明': '各笔 hold_days 相加，重叠仓位重复计算'},
    {'指标': '实际有仓交易日数', '数值': f'{held_trading_days:,} 天', '说明': '至少有一笔仓位，重叠仓位只计一次'},
    {'指标': '交易日持仓比例', '数值': f'{trading_day_exposure:.2%}', '说明': '实际有仓交易日数 / 总交易日时长'},
    {'指标': '累计仓位日比例', '数值': f'{gross_position_day_ratio:.2%}', '说明': '累计单笔持仓天数 / 总交易日；可能超过100%'},
    {'指标': '实际有仓自然日数', '数值': f'{held_calendar_days:,} 天', '说明': '包括隔夜、周末和节假日，重叠去重'},
    {'指标': '自然日持仓比例', '数值': f'{calendar_day_exposure:.2%}', '说明': '实际有仓自然日数 / 总自然日时长'},
    {'指标': '总净 PnL', '数值': f"{strategy_metrics_raw['total_net_pnl_bp']:+,.2f} bp", '说明': '所有交易 pnl_bp_net 之和'}
])

display(strategy_metrics)


,指标,数值,说明
0,首次入场日期,2020-07-28,策略实际开始持仓
1,最后退出日期,2026-06-16,策略最后一笔交易退出
2,总自然日时长,"2,150 天",首次入场至最后退出，首尾均包含
3,总交易日时长,"1,426 天",同一区间内的真实交易日数量
4,纳入信号日数,44 天,去重后的 signal_date 数量
5,完成交易数,82 笔,一个信号日可由多个因子产生多笔交易
6,累计单笔持仓天数,366 仓位日,各笔 hold_days 相加，重叠仓位重复计算
7,实际有仓交易日数,183 天,至少有一笔仓位，重叠仓位只计一次
8,交易日持仓比例,12.83%,实际有仓交易日数 / 总交易日时长
9,累计仓位日比例,25.67%,累计单笔持仓天数 / 总交易日；可能超过100%
